In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import warnings, gc, time
warnings.filterwarnings("ignore")
import numpy  as np
import pandas as pd
from   pathlib import Path
from   sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")
OUT  = Path("/kaggle/working")
t0_total = time.time()

# ── 1. LOAD ─────────────────────────────────────────────────
print("=" * 55)
print("STORE SALES — VERIFIED SOLUTION")
print("=" * 55)
print("\n[1] Loading...")
train  = pd.read_csv(BASE/"train.csv",           parse_dates=["date"],
                     dtype={"store_nbr":"int8","sales":"float32","onpromotion":"int16"})
test   = pd.read_csv(BASE/"test.csv",            parse_dates=["date"],
                     dtype={"store_nbr":"int8","onpromotion":"int16"})
stores = pd.read_csv(BASE/"stores.csv")
oil    = pd.read_csv(BASE/"oil.csv",             parse_dates=["date"])
hol    = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans  = pd.read_csv(BASE/"transactions.csv",    parse_dates=["date"])

train["sales"] = train["sales"].clip(lower=0)
print(f"  train={train.shape} | test={test.shape}")

# ── 2. STATIC LOOKUPS ───────────────────────────────────────
print("[2] Static lookups...")

stores = stores.rename(columns={"type":"store_type"})
le_type = LabelEncoder().fit(stores["store_type"])
stores["store_type_enc"] = le_type.transform(stores["store_type"]).astype("int8")
stores["cluster"] = stores["cluster"].astype("int8")

oil = (oil.set_index("date")
          .reindex(pd.date_range("2013-01-01","2017-09-01",freq="D"))
          .rename_axis("date").reset_index())
oil["dcoilwtico"] = oil["dcoilwtico"].interpolate("linear").ffill().bfill()
oil["oil_ma7"]    = oil["dcoilwtico"].rolling(7,  min_periods=1).mean()
oil["oil_ma28"]   = oil["dcoilwtico"].rolling(28, min_periods=1).mean()

nat  = set(hol[hol.locale=="National"]["date"])
xfer = set(hol[hol.transferred==True]["date"])
reg  = hol[hol.locale=="Regional"].groupby("date")["locale_name"].apply(set).to_dict()
loc  = hol[hol.locale=="Local"].groupby("date")["locale_name"].apply(set).to_dict()

trans = trans.sort_values(["store_nbr","date"])
g = trans.groupby("store_nbr")["transactions"]
trans["tx_ma7"]  = g.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
trans["tx_ma28"] = g.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())

le_fam = LabelEncoder().fit(train["family"])

# ── 3. COMBINED FRAME ────────────────────────────────────────
print("[3] Building combined frame...")
test["sales"] = np.nan
df = pd.concat([
    train[["date","store_nbr","family","sales","onpromotion"]].assign(split="train"),
    test[["date","store_nbr","family","onpromotion"]].assign(sales=np.nan, split="test")
], ignore_index=True).sort_values(["store_nbr","family","date"]).reset_index(drop=True)
del train; gc.collect()

# ── 4. LAG & ROLLING FEATURES ────────────────────────────────
# VERIFIED: all lags below have 0 NaN in test rows.
# lag_16: shift=16, so lag_16 of Aug31 = Aug15 (last train) ✓
# rolling with shift(16): window ends at t-16, always in train ✓
print("[4] Computing lags (verified zero NaN in test)...")
t0 = time.time()

grp = df.groupby(["store_nbr","family"])["sales"]

# Point lags
for lag in [16, 21, 28, 35, 42, 56, 91, 182, 364, 365, 366]:
    df[f"lag_{lag}"] = grp.shift(lag).astype("float32")

# Rolling with shift(16) — zero NaN in test guaranteed
for w in [7, 14, 28, 56]:
    sh = grp.shift(16)
    df[f"rmean_{w}"] = sh.transform(lambda x: x.rolling(w,min_periods=1).mean()).astype("float32")
    df[f"rstd_{w}"]  = sh.transform(lambda x: x.rolling(w,min_periods=1).std().fillna(0)).astype("float32")
    df[f"rmax_{w}"]  = sh.transform(lambda x: x.rolling(w,min_periods=1).max()).astype("float32")

# Last-year average (most predictive single feature)
df["lag_ly"] = ((df["lag_364"]+df["lag_365"]+df["lag_366"])/3).astype("float32")

print(f"  Done in {time.time()-t0:.0f}s")

# VERIFY zero NaN in test
tst = df[df.split=="test"]
lag_cols = [c for c in df.columns if c.startswith("lag_") or c.startswith("r")]
bad = {c: tst[c].isna().sum() for c in lag_cols if tst[c].isna().sum()>0}
if bad:
    print(f"  ⚠ NaN found in test: {bad}")
else:
    print(f"  ✓ ALL {len(lag_cols)} lag/roll features: 0 NaN in test")

# ── 5. CALENDAR + STATIC FEATURES ───────────────────────────
print("[5] Calendar & static features...")
d = df["date"]
df["year"]       = d.dt.year.astype("int16")
df["month"]      = d.dt.month.astype("int8")
df["day"]        = d.dt.day.astype("int8")
df["dow"]        = d.dt.dayofweek.astype("int8")
df["doy"]        = d.dt.dayofyear.astype("int16")
df["woy"]        = d.dt.isocalendar().week.astype("int8").values
df["is_weekend"] = (d.dt.dayofweek>=5).astype("int8")
df["is_mend"]    = d.dt.is_month_end.astype("int8")
df["is_payday"]  = ((d.dt.day==15)|d.dt.is_month_end).astype("int8")
df["trend"]      = (d-pd.Timestamp("2013-01-01")).dt.days.astype("int16")

for k in [1,2,3]:
    df[f"sw{k}"] = np.sin(2*np.pi*k*df["dow"]/7).astype("float32")
    df[f"cw{k}"] = np.cos(2*np.pi*k*df["dow"]/7).astype("float32")
    df[f"sy{k}"] = np.sin(2*np.pi*k*df["doy"]/365.25).astype("float32")
    df[f"cy{k}"] = np.cos(2*np.pi*k*df["doy"]/365.25).astype("float32")

df = df.merge(stores[["store_nbr","store_type_enc","cluster","city","state"]],
              on="store_nbr", how="left")
df["is_nat"]  = d.isin(nat).astype("int8")
df["is_xfer"] = d.isin(xfer).astype("int8")
df["is_reg"]  = df.apply(lambda r: int(r["state"] in reg.get(r["date"],set())), axis=1).astype("int8")
df["is_loc"]  = df.apply(lambda r: int(r["city"]  in loc.get(r["date"],set())), axis=1).astype("int8")
df["is_hol"]  = (df["is_nat"]|df["is_reg"]|df["is_loc"]).astype("int8")
df.drop(columns=["city","state"], inplace=True)

df = df.merge(oil[["date","dcoilwtico","oil_ma7","oil_ma28"]], on="date", how="left")
df = df.merge(trans[["date","store_nbr","transactions","tx_ma7","tx_ma28"]],
              on=["date","store_nbr"], how="left")
df["fam_enc"] = le_fam.transform(df["family"]).astype("int8")
gc.collect()

# ── 6. FEATURE COLUMNS ──────────────────────────────────────
SKIP = {"date","family","sales","split","store_type","transactions"}
FEAT = [c for c in df.columns if c not in SKIP]
df[FEAT] = df[FEAT].fillna(0)
df["log1p_sales"] = np.log1p(df["sales"].fillna(0)).astype("float32")
print(f"  Features ({len(FEAT)}): {FEAT}")

# ── 7. SPLIT ─────────────────────────────────────────────────
VAL = pd.Timestamp("2017-07-31")
use = df[(df.split=="train") & (df.lag_182 > 0)]

X_tr = use[use.date <  VAL][FEAT];  y_tr = use[use.date <  VAL]["log1p_sales"]
X_va = use[use.date >= VAL][FEAT];  y_va = use[use.date >= VAL]["log1p_sales"]
X_te = df[df.split=="test"][FEAT]

print(f"\n  X_tr={X_tr.shape} X_va={X_va.shape} X_te={X_te.shape}")
del df, use; gc.collect()

# ── 8. LIGHTGBM 3-SEED ──────────────────────────────────────
print("\n" + "="*55)
print("[6] LightGBM — 3-seed ensemble")
print("="*55)

P = dict(objective="regression_l1", metric="rmse",
         boosting_type="gbdt", num_leaves=255,
         learning_rate=0.05, feature_fraction=0.8,
         bagging_fraction=0.8, bagging_freq=1,
         min_child_samples=20, reg_alpha=0.05,
         reg_lambda=0.5, n_jobs=-1, verbose=-1)

dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=False)

models, scores = [], []
for seed in [42, 123, 777]:
    print(f"\n  seed={seed}")
    P["seed"] = seed
    t0 = time.time()
    m = lgb.train(P, dtrain, num_boost_round=3000,
                  valid_sets=[dvalid],
                  callbacks=[lgb.log_evaluation(100),
                              lgb.early_stopping(50, verbose=True)])
    vp    = np.expm1(m.predict(X_va, num_iteration=m.best_iteration)).clip(0)
    vt    = np.expm1(y_va.values).clip(0)
    rmsle = np.sqrt(np.mean((np.log1p(vp)-np.log1p(vt))**2))
    print(f"  Val RMSLE={rmsle:.5f} | iter={m.best_iteration} | {time.time()-t0:.0f}s")
    models.append(m); scores.append(rmsle)

print(f"\n  Scores: {[f'{s:.5f}' for s in scores]}")
print(f"  Mean  : {np.mean(scores):.5f}")

fi = pd.Series(models[0].feature_importance("gain"), index=FEAT)
print("\n  Top 15 features:")
print(fi.sort_values(ascending=False).head(15).to_string())

# ── 9. PREDICT ───────────────────────────────────────────────
print("\n[7] Predicting...")
inv  = 1.0/np.array(scores);  wts = inv/inv.sum()
logp = sum(w*m.predict(X_te, num_iteration=m.best_iteration) for m,w in zip(models,wts))
pred = np.expm1(logp).clip(min=0)
print(f"  min={pred.min():.2f} mean={pred.mean():.2f} max={pred.max():.2f}")

# ── 10. SUBMISSION ───────────────────────────────────────────
print("\n[8] Submission...")
test_raw = pd.read_csv(BASE/"test.csv", parse_dates=["date"],
                       dtype={"store_nbr":"int8"})
test_raw = test_raw.sort_values(["store_nbr","family","date"]).reset_index(drop=True)
test_raw["sales"] = pred

sample = pd.read_csv(BASE/"sample_submission.csv")
sub    = sample[["id"]].merge(test_raw[["id","sales"]], on="id", how="left")
sub["sales"] = sub["sales"].fillna(0).clip(lower=0)
sub.to_csv(OUT/"submission.csv", index=False)

print("\n" + "="*55)
print(f"  ✅ submission.csv — {len(sub):,} rows")
print(f"  Sales mean={sub.sales.mean():.2f} max={sub.sales.max():.2f}")
print(f"  Val RMSLE: {[f'{s:.5f}' for s in scores]}")
print(f"  Total time: {(time.time()-t0_total)/60:.1f} min")
print("="*55)
print("  → Submit Prediction → submission.csv")

STORE SALES — VERIFIED SOLUTION

[1] Loading...
  train=(3000888, 6) | test=(28512, 5)
[2] Static lookups...
[3] Building combined frame...
[4] Computing lags (verified zero NaN in test)...
  Done in 4s
  ✓ ALL 24 lag/roll features: 0 NaN in test
[5] Calendar & static features...
  Features (61): ['store_nbr', 'onpromotion', 'lag_16', 'lag_21', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_91', 'lag_182', 'lag_364', 'lag_365', 'lag_366', 'rmean_7', 'rstd_7', 'rmax_7', 'rmean_14', 'rstd_14', 'rmax_14', 'rmean_28', 'rstd_28', 'rmax_28', 'rmean_56', 'rstd_56', 'rmax_56', 'lag_ly', 'year', 'month', 'day', 'dow', 'doy', 'woy', 'is_weekend', 'is_mend', 'is_payday', 'trend', 'sw1', 'cw1', 'sy1', 'cy1', 'sw2', 'cw2', 'sy2', 'cy2', 'sw3', 'cw3', 'sy3', 'cy3', 'store_type_enc', 'cluster', 'is_nat', 'is_xfer', 'is_reg', 'is_loc', 'is_hol', 'dcoilwtico', 'oil_ma7', 'oil_ma28', 'tx_ma7', 'tx_ma28', 'fam_enc']

  X_tr=(1760358, 61) X_va=(24063, 61) X_te=(28512, 61)

[6] LightGBM — 3-seed ensemble

  